In [ ]:
# !ln -s /kaggle/input/datasets/alifamirudinstei/prjkcad-configs configs

In [ ]:
# import sys
# sys.path.append("prjkcad")

# Coreset Creation

In [ ]:
# Coreset Creation Cell
# Based on training-free coreset generation method : 

import os
from utils.data_utils import CoresetCreator
from utils.dual_seq import DualSeq

# Define settings
DATA_ROOT = "data/text2cad"

# Initialize CoresetCreator
creator = CoresetCreator(
    data_root=DATA_ROOT,
    source_data_type="text2cad",
    description_level="expert",
    p=0.005,
    k=100,
    model_name="bert-base-uncased",
    batch_size=16
)
# Create and save the coreset
coreset = creator.create_coreset(selection_method="closest")

print(f"Coreset Size: {len(coreset)}")

## Model Pruning Pipelines

In [ ]:
# Load configs
from utils.pipeline import load_config
from utils.pipeline.config import ModelConfig
base_cfg = load_config("configs/experiment_1apre.yaml")
# base_cfg = load_config("configs/sanity_c.yaml")
base_cfg.data.is_cmdonly = False
base_cfg.trainer.epochs = 10
max_new_cmds = base_cfg.model.max_new_cmds

from models import BaseModel

possible_model_configs = []

possible_encoders = ["bert"]
possible_d_models = [512] # Tied to encoders
possible_cmd_decoders = ["torch", "sdpa", "t5-small", "mamba"]
possible_args_decoders = ["torch", "sdpa", "t5-small", "mamba"]
possible_moe_types = ["OnFFN", "MoA"]
possible_moe_confs = ["Switch", "Mixtral", "DeepSeek"]

# Below brute force iteration needs to be simplified with optuna
for i, enc in enumerate(possible_encoders) :
    d_model = possible_d_models[i]
    for cmd_dec in possible_cmd_decoders :
        for args_dec in possible_args_decoders :
            # Case no MoE
            new_conf = ModelConfig(
                is_pretrained=False,
                encoder_type=enc,
                cmd_decoder_type=cmd_dec,
                args_decoder_type=args_dec,
                max_new_cmds = max_new_cmds 
            )

            possible_model_configs.append(new_conf)

            # Case MoE
            for moe_type in possible_moe_types :
                for moe_conf in possible_moe_confs :
                    new_conf = ModelConfig(
                        is_pretrained=False,
                        encoder_type=enc,
                        cmd_decoder_type=cmd_dec,
                        args_decoder_type=args_dec,
                        moe_type=moe_type,
                        moe_conf=moe_conf,
                        max_new_cmds = max_new_cmds
                    )

                    possible_model_configs.append(new_conf)

print(f"total_model_conf={len(possible_model_configs)}")

In [ ]:
# Optuna Architecture Search
# Resumes automatically if interrupted (SQLite checkpoint at out/optuna/optuna.db)
# Equivalent configurations are cached (out/optuna/config_cache.json)

from utils.pruning import run_optuna_study

study = run_optuna_study(
    base_cfg=base_cfg,
    coreset=coreset,
    n_trials=100,
    output_dir="out/optuna",
    study_name="prjkcad_arch_search",
)


In [ ]:
from utils.pipeline import merge_best_epochs
merge_best_epochs(
    out_dir="out/optuna"
)